## What makes a basketball team win?

The first time I watched Shai Gilgeous-Alexander play on TV, I was treated to the statistic that it was his `20`th consecutive game in which he had scored `20`+ points (this happened against the Suns on `November 28, 2025`).

Insane! A streak like that can be revolutionary for team performance, and I was not alone in this claim; leading into the playoffs, OKC were primed by the prediction markets to be favorites for the championship. However, despite SGA posting eye-popping playoff averages of `27.6` points and `7.9` assists per game, OKC were not able to repeat their championship-winning performance from their previous season - thanks to Victor Wembanyama & Co.

The stats above and the outcomes of the `2025-2026` NBA season (at the start of the Conference Finals, Knicks had a `6%` chance to win the championship) led me to think: what are the fundamental aspects of basketball that lead to a team win?

Let's investigate.

---
What I'm trying to solve for can be described as

`The probability of a team winning (%) = `

`(a * box_score_metric_1) + `

`(b * box_score_metric_2) + `

`(coefficient * however many box_score_metric_x variables my model says is relevant) + `

`luck`

To conduct this experiment, I will be using mathematical regression - or simply, the analysis of how inputs and outputs of a function/process/event are related to each other.

More specifically, I will be using the Elastic Net regression. I am using this variant because:
1. In a normal (linear) regression, all inputs are given importance.


"Many of my variables are correlated with each other, and I don't know ahead of time which ones actually matter. Elastic Net helps by automatically selecting the most useful variables while reducing the influence of redundant ones."

To redefine the question for code, let's make some definitions:
- Goal Statement: use box scores to infer the historical probability of a win
- Independent variable: win percentage, derived from box scores (target feature).
- Dependent variables: a variety of box score numbers, selected manually.
- Assumption #1: historical box stats contain useful information to prescribe the probability of a win.
- Assumption #2: the regression methodology is effective at identifying the most relevant variables that prescribe win probability.

___


Limitations:
An alternate test test would be 
---

In [20]:
from pathlib import Path
import time

import pandas as pd
from nba_api.stats.endpoints import leaguegamelog
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler

'''
The notebook keeps raw game logs locally, then builds the team-season table from those games.
'''

# Seasons included in the analysis.
seasons = ["2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]

# Local CSV files keep NBA.com downloads from running every time.
DATA_DIR = Path("data/nba")
DATA_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "regular_season": DATA_DIR / "nba_team_games_regular_season.csv",
    "playoffs": DATA_DIR / "nba_team_games_playoffs.csv",
}

Let's use data from the 2021-2026 season - `6` seasons, `30` teams, and `82` games/season/team.

`Sample size: 12300`

After fetching the data from the `nba_api` library, let's print it to see what it looks like.

In [23]:
# Gets one row per team-game from NBA.com for each season.
def fetch_games(season_type):
    all_seasons = []

    for season in seasons:
        print(f"Fetching {season} {season_type} games...")

        games = leaguegamelog.LeagueGameLog(
            player_or_team_abbreviation="T",
            season=season,
            season_type_all_star=season_type,
            timeout=60,
        ).get_data_frames()[0]

        games["SEASON"] = season
        games["SEASON_TYPE"] = season_type
        all_seasons.append(games)
        time.sleep(1)

    return pd.concat(all_seasons, ignore_index=True)


# Uses the local CSV if it exists; otherwise fetches the data once and saves it.
def load_or_fetch(path, fetch_fn):
    if path.exists():
        print(f"Loading local file: {path}")
        return pd.read_csv(path)

    data = fetch_fn()
    data.to_csv(path, index=False)
    return data


# Turns one row per team-game into one row per team-season.
def summarize_team_seasons(games):
    games = games.copy()

    # A team's blocks against are the opponent's blocks in the same game.
    opponent_totals = games.groupby("GAME_ID")[["BLK", "PF"]].transform("sum")
    games["BLKA"] = opponent_totals["BLK"] - games["BLK"]
    games["PFD"] = opponent_totals["PF"] - games["PF"]

    # Wins come directly from the game result column.
    games["WIN"] = games["WL"].eq("W").astype(int)

    team_columns = ["SEASON", "SEASON_TYPE", "TEAM_ID", "TEAM_ABBREVIATION", "TEAM_NAME"]
    stats_to_sum = [
        "MIN", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
        "OREB", "DREB", "REB", "AST", "STL", "BLK", "BLKA",
        "TOV", "PF", "PFD", "PTS", "PLUS_MINUS",
    ]

    record = games.groupby(team_columns, as_index=False).agg(
        GP=("GAME_ID", "count"),
        W=("WIN", "sum"),
    )
    totals = games.groupby(team_columns, as_index=False)[stats_to_sum].sum()
    summary = record.merge(totals, on=team_columns)

    # Rebuild the summary stats from the game rows.
    summary["L"] = summary["GP"] - summary["W"]
    summary["W_PCT"] = summary["W"] / summary["GP"]
    summary["FG_PCT"] = summary["FGM"] / summary["FGA"]
    summary["FG3_PCT"] = summary["FG3M"] / summary["FG3A"]
    summary["FT_PCT"] = summary["FTM"] / summary["FTA"]

    return summary

df_regular_season = load_or_fetch(
    files["regular_season"],
    lambda: fetch_games("Regular Season"),
)

df_playoffs = load_or_fetch(
    files["playoffs"],
    lambda: fetch_games("Playoffs"),
)

print("Regular-season team-game rows:", len(df_regular_season))
df_model_source = summarize_team_seasons(df_regular_season)
print(df_model_source.head())

Loading local file: data/nba/nba_team_games_regular_season.csv
Loading local file: data/nba/nba_team_games_playoffs.csv
Regular-season team-game rows: 12300
    SEASON     SEASON_TYPE     TEAM_ID TEAM_ABBREVIATION  \
0  2021-22  Regular Season  1610612737               ATL   
1  2021-22  Regular Season  1610612738               BOS   
2  2021-22  Regular Season  1610612739               CLE   
3  2021-22  Regular Season  1610612740               NOP   
4  2021-22  Regular Season  1610612741               CHI   

              TEAM_NAME  GP   W    MIN   FGM   FGA  ...   TOV    PF   PFD  \
0         Atlanta Hawks  82  43  19705  3401  7241  ...   972  1534  1668   
1        Boston Celtics  82  51  19905  3341  7167  ...  1118  1521  1592   
2   Cleveland Cavaliers  82  44  19730  3255  6940  ...  1181  1433  1638   
3  New Orleans Pelicans  82  36  19755  3294  7212  ...  1153  1612  1681   
4         Chicago Bulls  82  46  19730  3422  7127  ...  1053  1540  1489   

    PTS  PLUS_MINUS

Let's clean up the dataset by defining only what the model should see.


In [13]:
target = "W_PCT"

# These columns either identify the team, are the answer, or are too directly tied to the answer.
columns_to_skip = {
    "TEAM_ID", "TEAM_NAME", "TEAM_ABBREVIATION", "SEASON", "SEASON_TYPE",
    "W", "L", "W_PCT", "MIN", "GP", "G", "PTS", "FGM", "FTM", "FTA",
    "OREB", "DREB", "PF", "PLUS_MINUS",
}

columns_to_drop = [
    column
    for column in df_model_source.columns
    if column in columns_to_skip or "_RANK" in column
]

y = df_model_source[target]
X = df_model_source.drop(columns=columns_to_drop).select_dtypes(include="number")
df_model = pd.concat([y, X], axis=1).dropna()

print("Features used by the model:")
print(list(X.columns))


Features used by the model:
['FGA', 'FG3M', 'FG3A', 'REB', 'AST', 'STL', 'BLK', 'BLKA', 'TOV', 'PFD', 'FG_PCT', 'FG3_PCT', 'FT_PCT']


In [7]:
y = df_model["W_PCT"]
X = df_model.drop(columns=["W_PCT"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    cv=5,
    random_state=42,
    max_iter=100000,
)
model.fit(X_scaled, y)

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_,
})
coef_df["Abs_Coeff"] = coef_df["Coefficient"].abs()

important_features = (
    coef_df[(coef_df["Coefficient"] != 0) & (coef_df["Abs_Coeff"] >= 0.01)]
    .sort_values("Abs_Coeff", ascending=False)
)
important_features["Coefficient"] = important_features["Coefficient"].round(3)

print(important_features[["Feature", "Coefficient"]])


    Feature  Coefficient
3       REB        0.070
8       TOV       -0.052
10   FG_PCT        0.048
5       STL        0.045
0       FGA       -0.038
1      FG3M        0.022
11  FG3_PCT        0.022
7      BLKA       -0.020
9       PFD        0.019
12   FT_PCT        0.010


Win Rate (%) = c + 0.054 REB + 0.048 FG_PCT + ...

Ignoring assists, I can build a model from here.